In [2]:
pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   - --------------------


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [4]:
# chargement des données 
try:
    df = pd.read_csv('Student_Performance.csv')
    print("Dataset chargé avec succès.")
    print(f"Dimensions initiales : {df.shape}")
except FileNotFoundError:
    print("Erreur : Le fichier 'Student_Performance.csv' est introuvable.")
    exit()

Dataset chargé avec succès.
Dimensions initiales : (25000, 16)


In [5]:
# exploration et vérification de la qualité
print("\n--- Aperçu des premières lignes ---")
print(df.head())

print("\n--- Informations sur les colonnes et types de données ---")
df.info()

print("\n--- Valeurs manquantes par colonne ---")
print(df.isnull().sum())

print("\n--- Nombre de doublons ---")
print(df.duplicated().sum())


--- Aperçu des premières lignes ---
   student_id  age  gender school_type parent_education  study_hours  \
0           1   14    male      public    post graduate          3.1   
1           2   18  female      public         graduate          3.7   
2           3   17  female     private    post graduate          7.9   
3           4   16   other      public      high school          1.1   
4           5   16  female      public      high school          1.3   

   attendance_percentage internet_access travel_time extra_activities  \
0                   84.3             yes     <15 min              yes   
1                   87.8             yes     >60 min               no   
2                   65.5              no     <15 min               no   
3                   58.1              no   15-30 min               no   
4                   61.0             yes   30-60 min              yes   

  study_method  math_score  science_score  english_score  overall_score  \
0        notes  

In [6]:
# Suppression des doublons 
df.drop_duplicates(inplace=True)
print(f"\nDimensions après suppression des doublons : {df.shape}")

# La colonne 'student_id' est un identifiant unique, elle n'est pas utile pour la prédiction.
# On la supprime.
if 'student_id' in df.columns:
    df = df.drop('student_id', axis=1)
    print("Colonne 'student_id' supprimée.")


Dimensions après suppression des doublons : (15000, 16)
Colonne 'student_id' supprimée.


In [13]:
# valeurs manquantes
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include=['object', 'string']).columns

for col in num_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)

print("\nVérification des valeurs manquantes après imputation :")
print(df.isnull().sum().sum())


Vérification des valeurs manquantes après imputation :
0


In [14]:
# valeurs abérrentes
def cap_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # On applique le capping
    df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
    df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])
    return df

outlier_cols = ['study_hours', 'attendance_percentage', 'math_score', 'science_score', 'english_score', 'overall_score']
for col in outlier_cols:
    if col in df.columns:
        df = cap_outliers(df, col)

print("\nTraitement des valeurs aberrantes terminé.")


Traitement des valeurs aberrantes terminé.


In [9]:
# Colonnes catégorielles à encoder en One-Hot
cat_cols_to_encode = ['gender', 'school_type', 'parent_education', 'internet_access', 'travel_time', 'extra_activities', 'study_method']

# Mapping pour la colonne 'final_grade' (ordre alphabétique)
grade_mapping = {'a': 6, 'b': 5, 'c': 4, 'd': 3, 'e': 2, 'f': 1}
df['final_grade_encoded'] = df['final_grade'].map(grade_mapping)

# On supprime la colonne originale 'final_grade' et on garde l'encodée
df = df.drop('final_grade', axis=1)

# On applique le One-Hot Encoding
df = pd.get_dummies(df, columns=cat_cols_to_encode, drop_first=True)

print("\nEncodage des variables catégorielles terminé.")
print(f"Nouvelles dimensions après encodage : {df.shape}")


Encodage des variables catégorielles terminé.
Nouvelles dimensions après encodage : (15000, 26)


In [10]:
cols_to_scale = ['age', 'study_hours', 'attendance_percentage', 'math_score', 'science_score', 'english_score', 'overall_score']

scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

print("\nNormalisation des variables numériques terminée.")


Normalisation des variables numériques terminée.


In [11]:
df.to_csv('Student_Performance_Cleaned.csv', index=False)
print("\nDataset nettoyé et préparé sauvegardé sous le nom 'Student_Performance_Cleaned.csv'.")

# Affichage final
print("\n--- Aperçu du dataset final ---")
print(df.head())
print(f"\nDimensions finales : {df.shape}")


Dataset nettoyé et préparé sauvegardé sous le nom 'Student_Performance_Cleaned.csv'.

--- Aperçu du dataset final ---
        age  study_hours  attendance_percentage  math_score  science_score  \
0 -1.452929    -0.533685               0.646376   -1.007446      -0.397170   
1  0.893912    -0.257489               0.889420   -0.295172       0.240109   
2  0.307202     1.675885              -0.659120    1.005087       1.486132   
3 -0.279509    -1.454339              -1.172985   -0.926180      -1.724042   
4 -0.279509    -1.362274              -0.971606   -2.623209      -1.476740   

   english_score  overall_score  final_grade_encoded  gender_male  \
0      -0.321648      -0.575191                    2         True   
1       0.052287      -0.143093                    3        False   
2       0.742630       1.348172                    5        False   
3      -0.431911      -1.181182                    2        False   
4      -1.616041      -2.034839                    1        False  